# Classification Workflow

This notebook follows a practical classification workflow across multiple data types: image labels, tabular passenger records, and raw email text.

## Workflow Overview

**Objective:** build classifiers that predict discrete labels and evaluate them with task-appropriate metrics.

**Datasets:** MNIST digits, Titanic survival data, and Apache SpamAssassin emails.

**Primary metrics:** accuracy for balanced digit classification; precision, recall, F1, and confusion matrices for survival/spam tasks where error types matter.

**Validation strategy:** tune models on training/validation data, keep test data for final checks, and use stratified splits where class balance matters.

**Production mindset:** preprocessing is placed in pipelines so the same text cleaning, vectorization, imputation, encoding, and scaling steps are reused consistently.

## Stage 0 - Environment and Runtime Controls

Define imports, paths, reproducibility constants, and switches for expensive MNIST runs. This makes the notebook runnable on a laptop while still supporting full experiments.

In [ ]:
from pathlib import Path
import email
import email.parser
import email.policy
import html
import re
import tarfile
import urllib.request

import numpy as np
import pandas as pd
from scipy.ndimage import shift
from scipy.stats import randint

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

# Full MNIST KNN runs can be slow. Set this to True for the full exercise run.
RUN_EXPENSIVE_MNIST = False
MNIST_GRID_TRAIN_SIZE = None if RUN_EXPENSIVE_MNIST else 10_000
MNIST_FINAL_TRAIN_SIZE = None if RUN_EXPENSIVE_MNIST else 20_000
MNIST_AUGMENT_TRAIN_SIZE = None if RUN_EXPENSIVE_MNIST else 8_000
MNIST_TEST_SIZE = None if RUN_EXPENSIVE_MNIST else 2_000

def first_n(X, y, n):
    if n is None:
        return X, y
    return X[:n], y[:n]


def describe_scores(scores):
    return pd.Series(scores).describe()

## Stage 1 - Load and Split MNIST

Load image features as NumPy arrays and keep the standard train/test split. The dataset is used for nearest-neighbor classification and augmentation tests.

MNIST already has the standard split used in the chapter: first 60,000 rows for training and last 10,000 rows for testing.

In [2]:
mnist = fetch_openml("mnist_784", as_frame=False, parser="auto")
X, y = mnist.data.astype(np.uint8), mnist.target.astype(str)

X_train, X_test = X[:60_000], X[60_000:]
y_train, y_test = y[:60_000], y[60_000:]

X_train.shape, X_test.shape, y_train[:5]

((60000, 784), (10000, 784), array(['5', '0', '4', '1', '9'], dtype='<U1'))

## Stage 2 - MNIST KNN Model Selection

Tune a KNN classifier with cross-validation. The goal is to establish a strong nonparametric image-classification baseline.

This experiment uses `KNeighborsClassifier` and a grid search over `weights` and `n_neighbors`.

For a quick run, the grid search below uses the first `MNIST_GRID_TRAIN_SIZE` rows. Set `RUN_EXPENSIVE_MNIST = True` in the setup cell to search/train/evaluate on the full dataset.

In [3]:
X_grid, y_grid = first_n(X_train, y_train, MNIST_GRID_TRAIN_SIZE)

knn_param_grid = {
    "weights": ["uniform", "distance"],
    "n_neighbors": [3, 4, 5],
}

knn_grid_search = GridSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    knn_param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
knn_grid_search.fit(X_grid, y_grid)

knn_grid_search.best_params_, knn_grid_search.best_score_

Fitting 3 folds for each of 6 candidates, totalling 18 fits


({'n_neighbors': 4, 'weights': 'distance'}, 0.9397994088551026)

In [4]:
X_knn_fit, y_knn_fit = first_n(X_train, y_train, MNIST_FINAL_TRAIN_SIZE)
X_knn_eval, y_knn_eval = first_n(X_test, y_test, MNIST_TEST_SIZE)

best_knn_clf = KNeighborsClassifier(**knn_grid_search.best_params_, n_jobs=-1)
best_knn_clf.fit(X_knn_fit, y_knn_fit)

knn_test_pred = best_knn_clf.predict(X_knn_eval)
knn_test_accuracy = accuracy_score(y_knn_eval, knn_test_pred)
knn_test_accuracy

0.94

If you run the full version, `weights="distance"` and `n_neighbors=4` are commonly strong settings for MNIST KNN and should get around or above the target accuracy.

## Stage 3 - Data Augmentation for Robustness

Generate shifted versions of digit images to make the classifier less sensitive to small translations.

We create four extra copies of every training image:

- shifted left by 1 pixel
- shifted right by 1 pixel
- shifted up by 1 pixel
- shifted down by 1 pixel

This grows the training set and teaches the model that a digit is still the same digit if it moves slightly.

In [5]:
def shift_image(image, dx, dy):
    # Shift a flattened 28x28 MNIST image by dx columns and dy rows.
    image_2d = image.reshape(28, 28)
    shifted_image = shift(image_2d, shift=[dy, dx], cval=0, mode="constant", order=0)
    return shifted_image.reshape(-1)


# Quick sanity check: this returns one flattened 784-pixel image.
shift_image(X_train[0], dx=1, dy=0).shape

(784,)

In [6]:
def augment_mnist_by_shifting(X, y):
    shifted_images = [X]
    shifted_labels = [y]

    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        shifted = np.array([shift_image(image, dx=dx, dy=dy) for image in X], dtype=X.dtype)
        shifted_images.append(shifted)
        shifted_labels.append(y.copy())

    X_augmented = np.vstack(shifted_images)
    y_augmented = np.concatenate(shifted_labels)

    rng = np.random.default_rng(RANDOM_STATE)
    shuffled_indices = rng.permutation(len(X_augmented))
    return X_augmented[shuffled_indices], y_augmented[shuffled_indices]


X_aug_base, y_aug_base = first_n(X_train, y_train, MNIST_AUGMENT_TRAIN_SIZE)
X_train_augmented, y_train_augmented = augment_mnist_by_shifting(X_aug_base, y_aug_base)

X_train_augmented.shape, y_train_augmented.shape

((40000, 784), (40000,))

In [7]:
augmented_knn_clf = KNeighborsClassifier(**knn_grid_search.best_params_, n_jobs=-1)
augmented_knn_clf.fit(X_train_augmented, y_train_augmented)

augmented_pred = augmented_knn_clf.predict(X_knn_eval)
augmented_accuracy = accuracy_score(y_knn_eval, augmented_pred)

pd.Series({
    "before_augmentation_accuracy": knn_test_accuracy,
    "after_augmentation_accuracy": augmented_accuracy,
})

before_augmentation_accuracy    0.940
after_augmentation_accuracy     0.936
dtype: float64

## Stage 4 - Tabular Classification Pipeline

Build a full tabular pipeline with numeric/categorical preprocessing, model tuning, validation metrics, and a submission-style output table.

The goal is to predict the `Survived` column from the passenger information.

The official Kaggle test set does not include labels, so we split the provided `train.csv` into our own training and validation sets. Then we also create prediction output for Kaggle's unlabeled `test.csv`.

In [8]:
TITANIC_URL = "https://homl.info/titanic.tgz"
TITANIC_DIR = DATA_DIR / "titanic"
TITANIC_TGZ = DATA_DIR / "titanic.tgz"


def fetch_titanic_data():
    TITANIC_DIR.mkdir(exist_ok=True)
    train_csv = TITANIC_DIR / "train.csv"
    test_csv = TITANIC_DIR / "test.csv"
    if not train_csv.is_file() or not test_csv.is_file():
        urllib.request.urlretrieve(TITANIC_URL, TITANIC_TGZ)
        with tarfile.open(TITANIC_TGZ) as tar:
            tar.extractall(path=DATA_DIR)
    return pd.read_csv(train_csv), pd.read_csv(test_csv)


titanic_train_full, titanic_test = fetch_titanic_data()
titanic_train_full.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
X_titanic = titanic_train_full.drop(columns="Survived")
y_titanic = titanic_train_full["Survived"].copy()

X_titanic_train, X_titanic_valid, y_titanic_train, y_titanic_valid = train_test_split(
    X_titanic,
    y_titanic,
    test_size=0.2,
    stratify=y_titanic,
    random_state=RANDOM_STATE,
)

titanic_num_attribs = ["Age", "SibSp", "Parch", "Fare"]
titanic_cat_attribs = ["Pclass", "Sex", "Embarked"]

X_titanic_train.shape, X_titanic_valid.shape

((712, 11), (179, 11))

In [10]:
titanic_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

titanic_cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)

titanic_preprocessing = ColumnTransformer([
    ("num", titanic_num_pipeline, titanic_num_attribs),
    ("cat", titanic_cat_pipeline, titanic_cat_attribs),
])

titanic_pipeline = Pipeline([
    ("preprocessing", titanic_preprocessing),
    ("forest", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

titanic_cv_scores = cross_val_score(
    titanic_pipeline,
    X_titanic_train,
    y_titanic_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

describe_scores(titanic_cv_scores)

count    5.000000
mean     0.788072
std      0.060742
min      0.692308
25%      0.776224
50%      0.795775
75%      0.823944
max      0.852113
dtype: float64

In [11]:
titanic_param_distributions = {
    "forest__n_estimators": randint(100, 400),
    "forest__max_leaf_nodes": randint(8, 60),
    "forest__max_features": ["sqrt", "log2", None],
    "forest__min_samples_leaf": randint(1, 8),
}

titanic_search = RandomizedSearchCV(
    titanic_pipeline,
    titanic_param_distributions,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

titanic_search.fit(X_titanic_train, y_titanic_train)

titanic_search.best_score_, titanic_search.best_params_

Fitting 5 folds for each of 20 candidates, totalling 100 fits


(0.8315571752191471,
 {'forest__max_features': 'sqrt',
  'forest__max_leaf_nodes': 51,
  'forest__min_samples_leaf': 1,
  'forest__n_estimators': 148})

In [12]:
titanic_valid_pred = titanic_search.best_estimator_.predict(X_titanic_valid)

pd.Series({
    "validation_accuracy": accuracy_score(y_titanic_valid, titanic_valid_pred),
    "validation_precision": precision_score(y_titanic_valid, titanic_valid_pred),
    "validation_recall": recall_score(y_titanic_valid, titanic_valid_pred),
    "validation_f1": f1_score(y_titanic_valid, titanic_valid_pred),
})

validation_accuracy     0.810056
validation_precision    0.818182
validation_recall       0.652174
validation_f1           0.725806
dtype: float64

In [13]:
print(classification_report(y_titanic_valid, titanic_valid_pred))
confusion_matrix(y_titanic_valid, titanic_valid_pred)

              precision    recall  f1-score   support

           0       0.81      0.91      0.85       110
           1       0.82      0.65      0.73        69

    accuracy                           0.81       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.81      0.81      0.81       179



array([[100,  10],
       [ 24,  45]])

In [14]:
titanic_final_model = clone(titanic_search.best_estimator_)
titanic_final_model.fit(X_titanic, y_titanic)

titanic_test_predictions = titanic_final_model.predict(titanic_test)

titanic_submission = pd.DataFrame({
    "PassengerId": titanic_test["PassengerId"],
    "Survived": titanic_test_predictions,
})

titanic_submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


## Stage 5 - Text Classification Pipeline

Parse raw emails, clean text, vectorize words, compare classifiers, and evaluate spam detection with precision/recall/F1.

We download the public Apache SpamAssassin ham/spam corpus, parse the emails, convert each email into text, vectorize the words, and train classifiers.

The vectorizer uses `binary=True`, so each feature means word present or absent, as a compact bag-of-words representation.

In [15]:
SPAMASSASSIN_ROOT = "https://spamassassin.apache.org/old/publiccorpus/"
SPAM_DIR = DATA_DIR / "spamassassin"
SPAM_FILES = ["20030228_easy_ham.tar.bz2", "20030228_spam.tar.bz2"]


def fetch_spamassassin_data():
    SPAM_DIR.mkdir(exist_ok=True)
    for filename in SPAM_FILES:
        archive_path = SPAM_DIR / filename
        if not archive_path.is_file():
            urllib.request.urlretrieve(SPAMASSASSIN_ROOT + filename, archive_path)
        extracted_dir = SPAM_DIR / filename.replace("20030228_", "").replace(".tar.bz2", "")
        if not extracted_dir.exists():
            with tarfile.open(archive_path, "r:bz2") as tar:
                tar.extractall(path=SPAM_DIR)


def load_email(filepath):
    with open(filepath, "rb") as f:
        return email.parser.BytesParser(policy=email.policy.default).parse(f)


fetch_spamassassin_data()

ham_dir = SPAM_DIR / "easy_ham"
spam_dir = SPAM_DIR / "spam"

ham_paths = sorted(path for path in ham_dir.iterdir() if path.is_file() and path.name != "cmds")
spam_paths = sorted(path for path in spam_dir.iterdir() if path.is_file() and path.name != "cmds")

ham_emails = [load_email(path) for path in ham_paths]
spam_emails = [load_email(path) for path in spam_paths]

len(ham_emails), len(spam_emails)

(2500, 500)

In [16]:
X_spam = np.array(ham_emails + spam_emails, dtype=object)
y_spam = np.array([0] * len(ham_emails) + [1] * len(spam_emails))

X_spam_train, X_spam_test, y_spam_train, y_spam_test = train_test_split(
    X_spam,
    y_spam,
    test_size=0.2,
    stratify=y_spam,
    random_state=RANDOM_STATE,
)

X_spam_train.shape, X_spam_test.shape, y_spam_train.mean(), y_spam_test.mean()

((2400,), (600,), 0.16666666666666666, 0.16666666666666666)

In [17]:
def html_to_plain_text(html_content):
    text = re.sub(r"<head.*?>.*?</head>", " ", html_content, flags=re.M | re.S | re.I)
    text = re.sub(r"<a\s.*?>", " HYPERLINK ", text, flags=re.M | re.S | re.I)
    text = re.sub(r"<.*?>", " ", text, flags=re.M | re.S)
    text = html.unescape(text)
    return text


def email_to_text(message):
    html_content = None
    for part in message.walk():
        content_type = part.get_content_type()
        if content_type not in ("text/plain", "text/html"):
            continue
        try:
            content = part.get_content()
        except Exception:
            payload = part.get_payload(decode=True)
            if payload is None:
                continue
            content = payload.decode(errors="replace")
        if content_type == "text/plain":
            return content
        html_content = content
    if html_content:
        return html_to_plain_text(html_content)
    return ""


class EmailToCleanTextTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lowercase=True, replace_urls=True, replace_numbers=True, remove_punctuation=True):
        self.lowercase = lowercase
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.remove_punctuation = remove_punctuation

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cleaned_texts = []
        for message in X:
            text = email_to_text(message)
            if self.lowercase:
                text = text.lower()
            if self.replace_urls:
                text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
            if self.replace_numbers:
                text = re.sub(r"\b\d+(?:\.\d+)?\b", " NUMBER ", text)
            if self.remove_punctuation:
                text = re.sub(r"[^a-zA-Z_]+", " ", text)
            cleaned_texts.append(text)
        return np.array(cleaned_texts, dtype=object)


email_to_text(ham_emails[0])[:300]

"    Date:        Wed, 21 Aug 2002 10:54:46 -0500\n    From:        Chris Garrigues <cwg-dated-1030377287.06fa6d@DeepEddy.Com>\n    Message-ID:  <1029945287.4797.TMDA@deepeddy.vircio.com>\n\n\n  | I can't reproduce this error.\n\nFor me it is very repeatable... (like every time, without fail).\n\nThis is the "

In [19]:
def make_spam_pipeline(classifier):
    return Pipeline([
        ("email_to_text", EmailToCleanTextTransformer()),
        ("word_vectorizer", CountVectorizer(binary=True, min_df=2, max_features=20_000)),
        ("classifier", classifier),
    ])


spam_pipelines = {
    "logistic_regression": make_spam_pipeline(LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    "multinomial_nb": make_spam_pipeline(MultinomialNB()),
    "linear_svc": make_spam_pipeline(LinearSVC(random_state=RANDOM_STATE, max_iter=5_000)),
}

spam_cv_results = []
for name, pipeline in spam_pipelines.items():
    f1_scores = cross_val_score(
        pipeline,
        X_spam_train,
        y_spam_train,
        cv=3,
        scoring="f1",
        n_jobs=-1,
    )
    spam_cv_results.append({
        "model": name,
        "mean_f1": f1_scores.mean(),
        "std_f1": f1_scores.std(),
    })

spam_cv_results = pd.DataFrame(spam_cv_results).sort_values("mean_f1", ascending=False)
spam_cv_results

,model,mean_f1,std_f1
1,multinomial_nb,0.968215,0.009749
0,logistic_regression,0.958603,0.018087
2,linear_svc,0.947100,0.021173


In [20]:
best_spam_model_name = spam_cv_results.iloc[0]["model"]
best_spam_pipeline = spam_pipelines[best_spam_model_name]
best_spam_pipeline.fit(X_spam_train, y_spam_train)

spam_test_pred = best_spam_pipeline.predict(X_spam_test)

pd.Series({
    "best_model": best_spam_model_name,
    "test_accuracy": accuracy_score(y_spam_test, spam_test_pred),
    "test_precision": precision_score(y_spam_test, spam_test_pred),
    "test_recall": recall_score(y_spam_test, spam_test_pred),
    "test_f1": f1_score(y_spam_test, spam_test_pred),
})

best_model        multinomial_nb
test_accuracy           0.981667
test_precision          0.978495
test_recall                 0.91
test_f1                 0.943005
dtype: object

In [21]:
print(classification_report(y_spam_test, spam_test_pred, target_names=["ham", "spam"]))
confusion_matrix(y_spam_test, spam_test_pred)

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       500
        spam       0.98      0.91      0.94       100

    accuracy                           0.98       600
   macro avg       0.98      0.95      0.97       600
weighted avg       0.98      0.98      0.98       600



array([[498,   2],
       [  9,  91]])

## Workflow Summary and Next Steps

Summarize the classification workflows, metrics, and practical extensions such as calibration, threshold tuning, and stronger text features.

The classification experiments implemented above:

1. `KNeighborsClassifier` for MNIST with grid search
2. MNIST data augmentation using one-pixel shifts
3. Titanic survival classifier with preprocessing and hyperparameter search
4. Spam classifier with email parsing, text cleaning, word-vectorization, and model comparison

For the exact full MNIST runs, set `RUN_EXPENSIVE_MNIST = True` in the setup cell and rerun the MNIST sections.